# Splitting data in train, validation, test

### Import libraries and tidied data

In [2]:
import scipy.sparse as sparse
import numpy as np

sparse_matrix = sparse.load_npz('data/tidied_360k/user_artist_matrix.npz')
users = np.load('data/tidied_360k/users.npy', allow_pickle=True)
artists = np.load('data/tidied_360k/artists.npy', allow_pickle=True)
artist_labels = np.load('data/tidied_360k/artist_labels.npy', allow_pickle=True)

In [3]:
print(artists[3])

0003ccd0-27cb-4e5d-9fea-a44c8111c1d9


In [4]:
print(users[0])

000215d3060a5b0ab7b3c415d49ec579100d4c87


In [5]:
print(sparse_matrix)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 964617 stored elements and shape (20000, 66231)>
  Coords	Values
  (0, 1117)	15
  (0, 1279)	15
  (0, 4719)	176
  (0, 7040)	97
  (0, 7447)	9
  (0, 9784)	16
  (0, 12048)	28
  (0, 13010)	30
  (0, 13193)	78
  (0, 13748)	33
  (0, 14529)	93
  (0, 14924)	49
  (0, 17146)	27
  (0, 17415)	10
  (0, 18747)	7
  (0, 18842)	7
  (0, 18970)	47
  (0, 19585)	28
  (0, 21117)	10
  (0, 24664)	12
  (0, 26123)	157
  (0, 26463)	9
  (0, 33144)	17
  (0, 34231)	19
  (0, 36746)	19
  :	:
  (19999, 33054)	193
  (19999, 36455)	2912
  (19999, 36975)	470
  (19999, 37169)	67
  (19999, 41890)	178
  (19999, 42853)	246
  (19999, 43213)	32
  (19999, 43441)	50
  (19999, 44818)	470
  (19999, 45031)	49
  (19999, 46320)	57
  (19999, 47648)	277
  (19999, 48959)	422
  (19999, 50789)	23
  (19999, 51007)	25
  (19999, 52719)	497
  (19999, 53275)	24
  (19999, 53288)	150
  (19999, 59741)	176
  (19999, 60864)	63
  (19999, 61858)	25
  (19999, 62610)	58
  (19999, 64682)	96
  (19

### Decide train/test/validation split

In [6]:
splits = {
    "train": 0.7,
    "validation": 0.2,
    "test": 0.1
}

### Split within a user's data

This way the same user's data will be available to test on

In [7]:
rng = np.random.default_rng(1235)

sparse_matrix.sum_duplicates() # eliminate duplicates by adding them together
split_labels = np.zeros(sparse_matrix.nnz, dtype=int)

for u in range(sparse_matrix.shape[0]): # iterate through each user in the sparse matrix
    # in compressed matrix data, finds all the non-zero data corresponding to this row
    start, end = sparse_matrix.indptr[u], sparse_matrix.indptr[u + 1]
    # number of artists in the data
    num_artists = end - start
    # random order
    order = rng.permutation(num_artists)
    # store the number of artists in each split
    num_artists_validation = int(splits["validation"] * num_artists)
    num_artists_test = int(splits["test"] * num_artists)
    # assign labels for validation and test (train is zero by default from np.zeros)
    for i in range(num_artists_validation):
        split_labels[start + order[i]] = 1
    for i in range(num_artists_test):
        split_labels[start + order[num_artists_validation + i]] = 2

def isolate_split(matrix, labels, label):
    # copy the sparse matrix
    split = matrix.copy()
    split.data[labels != label] = 0 # ignore cells from other splits
    split.eliminate_zeros() # remove those cells
    return split

train = isolate_split(sparse_matrix, split_labels, 0)
validation = isolate_split(sparse_matrix, split_labels, 1)
test = isolate_split(sparse_matrix, split_labels, 2)

In [8]:
rows, cols, values = sparse.find(sparse_matrix)
max_idx = np.argmax(values)

best_row = rows[max_idx]
best_col = cols[max_idx]
max_value = values[max_idx]

In [14]:
print(validation)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 184738 stored elements and shape (20000, 66231)>
  Coords	Values
  (0, 7447)	9
  (0, 9784)	16
  (0, 13193)	78
  (0, 14924)	49
  (0, 26463)	9
  (0, 38498)	16
  (0, 55898)	12
  (0, 60871)	42
  (0, 61517)	28
  (1, 12252)	174
  (1, 21073)	194
  (1, 21157)	218
  (1, 26621)	172
  (1, 28745)	148
  (1, 40607)	373
  (1, 42751)	456
  (1, 47987)	168
  (1, 63920)	174
  (2, 2821)	29
  (2, 5940)	22
  (2, 9626)	66
  (2, 16710)	45
  (2, 17893)	30
  (2, 26840)	20
  (2, 37010)	101
  :	:
  (19997, 2172)	13
  (19997, 12287)	18
  (19997, 21729)	9
  (19997, 25522)	12
  (19997, 38522)	12
  (19997, 51539)	9
  (19997, 53528)	10
  (19997, 61419)	10
  (19998, 6355)	31
  (19998, 13849)	38
  (19998, 14466)	35
  (19998, 19043)	46
  (19998, 22037)	54
  (19998, 28840)	24
  (19998, 40750)	28
  (19998, 49116)	36
  (19999, 1873)	36
  (19999, 1943)	88
  (19999, 11038)	170
  (19999, 23991)	1099
  (19999, 27213)	27
  (19999, 33054)	193
  (19999, 37169)	67
  (19999

In [15]:
(train.getnnz(axis=0) == 0).sum()

np.int64(8497)